<!-- ## Deep Residual Learning for Image Recognition -->
## 图像识别的深度残差学习


### 核心背景与问题
深层神经网络虽在视觉识别任务中潜力巨大，但存在训练难度随深度增加而显著上升的问题，传统深层网络易出现梯度消失 / 爆炸，导致模型难以优化且性能不升反降。
### 核心创新：残差学习框架
作者提出残差学习（Residual Learning） 范式，核心改进如下：
- 重构网络层目标：不再让每层学习无参考的原始映射，而是学习相对于输入的残差函数（即输出 = 输入 + 残差映射）。
- 引入 shortcut 连接（捷径连接）：直接跳过一个或多个卷积层，将输入信息直接传递到后续层，既简化梯度传播，又保留低层特征。
### 关键实验结果
- ImageNet 数据集：
  1. 实现最深 152 层的残差网络（ResNet），深度是 VGGNet 的 8 倍，但计算复杂度更低。
  2. 单模型性能优异，集成模型在 ImageNet 测试集上达到 3.57% 的错误率，斩获 2015 年 ILSVRC 图像分类任务冠军。
- CIFAR-10 数据集：
  - 验证了 100 层和 1000 层残差网络的有效性，证明残差结构可支撑超深层网络的稳定训练。
- 多任务拓展：
  1. 凭借超深表征能力，在 COCO 目标检测数据集上实现 28% 的相对性能提升。
  2. 成为 ILSVRC & COCO 2015 竞赛多个任务的基础架构，包揽 ImageNet 检测、ImageNet 定位、COCO 检测、COCO 分割四项任务冠军。
### 核心贡献
- 突破深层网络训练瓶颈，证明残差结构可让超深层网络（千层级）高效优化。
- 提供了兼顾深度与效率的网络设计范式，ResNet 成为后续计算机视觉任务（分类、检测、分割等）的基础架构。
- 验证了 “深度是视觉识别核心要素” 的观点，为后续深层网络研究奠定基础。

## 引言 Introduction

深度卷积神经网络为图像分类带来了一系列突破。
深度网络以端到端的多层方式自然地集成了低/中/高级特征和分类器，并且特征的“层次”可以通过堆叠层数（深度）来丰富。
最近的证据表明网络深度至关重要，并且具有挑战性的 ImageNet 数据集上的领先结果都利用了“非常深”的模型，深度为16到30。许多其他非平凡的视觉识别任务也从非常深的模型中大大受益。

<div style="background-color: white; width: 80%; margin:auto; text-align: center;">
    <image src="./assets/block.png" />
    <div style="font-size: 12px; color: gray;">图 1. 一个残差块的示意图。</div>
</div>

残差连接的核心代码:
```python
def forward(self, x: Tensor) -> Tensor:
    identity = x  # SOL 保存输入作为残差连接的基础
    # NT: 第一个卷积层
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    # NT: 第二个卷积层
    out = self.conv2(out)
    out = self.bn2(out)
    out = self.relu(out)
    # NT: 第三个卷积层
    out = self.conv3(out)
    out = self.bn3(out)
    # NT: 下采样层（如果存在）
    if self.downsample is not None:
        identity = self.downsample(x)  # SOL: 如果需要，调整输入维度以匹配输出
    # SOL: 残差连接
    out += identity
    out = self.relu(out)  # SOL: 最终激活函数在残差连接后应用
    return out
```

## 深度残差学习 Deep Residual Learning

<div style="background-color: white; width: 30%; margin:auto; text-align: center;">
        <image src="./assets/arch.png" style="max-width: 100%;" />
    <div style="font-size: 12px; color: gray;">图 2. ResNet 的网络结构示意图。</div>
</div>

- 线性层:
    1. VGG: (512, 7, 7) -> Flatten -> (25088,) -> FC(4096) -> FC(4096) -> FC(1000)
    2. ResNet: (2048, 7, 7) -> Global Average Pooling -> (2048,) -> FC(1000)